# MLE and Linear Regression

How does a *line* generate *points*? Like a road generates houses: close to it, rarely far — a vertical Gaussian at each $x_i$, centered on the line. With that generative story, "which line most likely produced these points?" becomes a computable maximum-likelihood question — and the algebra collapses into a famous answer: maximizing the likelihood is **exactly minimizing the sum of squared distances**. Maximum likelihood *is* least-squares linear regression. It is based on the sixth video of Lesson 6 (Point Estimation).

## 1. Introduction

The MLE recipe for machine learning: given data and candidate models, compute $P(\text{data} \mid \text{model})$ for each and crown the maximizer. This notebook runs that recipe on the model family that launched machine learning — lines $y = mx + b$ — and derives the connection promised since the popcorn video.

After completing this notebook you will be able to:

- State the generative model of regression: points sampled from Gaussians centered on the line (the road-and-houses picture).
- Write the likelihood of a dataset given a line and simplify it: constants drop, logs turn products into sums, and the sign flips maximization into minimization.
- Prove and verify that **maximum likelihood = least squares**.
- Reproduce the video's three-line contest and check the winner against `np.polyfit`.

## 2. Theory

### Lines that generate points: the road and the houses

Think of the line as a **road**; points are **houses**, likely built close to the road, unlikely far from it. Formally: at each location $x_i$, center a Gaussian (mean $0$, standard deviation $1$) *vertically* on the line's value $mx_i + b$, and sample the observed $y_i$ from it. Five points → five Gaussians on the line → five samples.

The best line is the one that **most likely produced the points** — and, coincidentally (not!), it will be the least-squares fit.

### The likelihood of one point

Let $d_i$ be the vertical distance between point $i$ and the line. With unit-Gaussian noise, the likelihood of generating that point is the Gaussian density at distance $d_i$:

$$
\frac{1}{\sqrt{2\pi}}\, e^{-\frac12 d_i^2}
$$

### The likelihood of all points — and the collapse

Independent points multiply:

$$
L = \prod_{i=1}^{5} \frac{1}{\sqrt{2\pi}} e^{-\frac12 d_i^2}
$$

Now simplify in three moves:

1. **Drop constants:** the $\frac{1}{\sqrt{2\pi}}$ factors don't depend on the line — forget them.
2. **Factor the exponentials:** $\prod e^{-\frac12 d_i^2} = e^{-\frac12 \sum d_i^2}$.
3. **Take the log** (same maximizer), discard the harmless $\tfrac12$: we are maximizing $-\sum d_i^2$.

Maximizing a negative is minimizing the positive:

$$
\text{maximize } L \quad\Longleftrightarrow\quad \text{minimize } \sum_{i} d_i^2
$$

**Minimizing the sum of squared distances is precisely the least-squares error.** Finding the line that most likely produced the points (maximum likelihood) is exactly linear regression's least-squares fit — the Gaussian noise model is the probabilistic soul of the squared-error loss.

### The three-line contest

With three candidate lines and their Gaussians drawn at each data point: the poorly-fitting lines put the observed points deep in their Gaussians' tails (tiny heights, tiny product), while the middle line keeps every point in the fat part of its Gaussians — highest likelihood, and visibly the best fit.

## 3. Worked Examples

### Example 1: two points, two lines, by hand

Points $(0, 1)$ and $(1, 3)$. Line A: $y = 2x + 1$ (distances $0, 0$) → likelihood $\left(\tfrac{1}{\sqrt{2\pi}}\right)^2 e^{0} \approx 0.159$. Line B: $y = x + 1$ (distances $0, 1$) → $\left(\tfrac{1}{\sqrt{2\pi}}\right)^2 e^{-1/2} \approx 0.0965$. Line A wins — and it has the smaller (zero) sum of squares.

### Example 2: the equivalence in numbers

For any two lines, (likelihood ratio) $= e^{-\frac12(\text{SSE}_1 - \text{SSE}_2)}$: comparing likelihoods and comparing sums of squared errors is the *same comparison* with a monotone change of scale. A line 2 units of SSE better is $e^1 \approx 2.72$ times more likely.

### Example 3: what the noise scale changes

Using noise std $\sigma$ instead of 1 multiplies the exponent by $\frac{1}{\sigma^2}$ — a *constant* with respect to $(m, b)$. The argmax line is unchanged: least squares is the MLE for every homoscedastic Gaussian noise level. ($\sigma$ matters for *how confident* we are, not for *which* line wins.)

## 4. Python Examples

First the imports (with the numpy 2.x compatibility shim used throughout this project).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# numpy < 2 fallback: np.trapezoid was called np.trapz
if not hasattr(np, "trapezoid"):
    np.trapezoid = np.trapz

rng = np.random.default_rng(42)

# the road generates the houses: y sampled from a Gaussian centered on the line
m_true, b_true = 1.2, 2.0
x = np.array([0.5, 1.5, 2.5, 3.5, 4.5])
y = m_true * x + b_true + rng.normal(0, 1, size=len(x))

def log_likelihood(m, b):
    d = y - (m * x + b)
    return np.sum(-0.5 * d**2 - 0.5 * np.log(2 * np.pi))

def sse(m, b):
    return np.sum((y - (m * x + b))**2)

### The three-line contest

In [ ]:
lines = {"model 1": (0.2, 4.5), "model 2": (1.2, 2.0), "model 3": (2.2, -0.5)}

print(f"{'line':<9} {'SSE':>8} {'log-likelihood':>15} {'likelihood':>12}")
for name, (m, b) in lines.items():
    print(f"{name:<9} {sse(m, b):>8.2f} {log_likelihood(m, b):>15.2f} "
          f"{np.exp(log_likelihood(m, b)):>12.2e}")
print("\nSmallest SSE and largest likelihood belong to the same line -- always.")

### Maximum likelihood = least squares, verified over a dense grid

Search thousands of lines twice — once maximizing the likelihood, once minimizing the SSE — and get the identical winner, which also matches `np.polyfit`.

In [ ]:
ms = np.linspace(-1, 3, 401)
bs = np.linspace(-2, 6, 401)
M, B = np.meshgrid(ms, bs)

D = y[None, None, :] - (M[..., None] * x[None, None, :] + B[..., None])
sse_grid = np.sum(D**2, axis=-1)
ll_grid = -0.5 * sse_grid - len(x) / 2 * np.log(2 * np.pi)

i_ll = np.unravel_index(np.argmax(ll_grid), ll_grid.shape)
i_sse = np.unravel_index(np.argmin(sse_grid), sse_grid.shape)
fit = np.polyfit(x, y, 1)

print(f"argmax likelihood: m = {M[i_ll]:.3f}, b = {B[i_ll]:.3f}")
print(f"argmin SSE:        m = {M[i_sse]:.3f}, b = {B[i_sse]:.3f}")
print(f"np.polyfit:        m = {fit[0]:.3f}, b = {fit[1]:.3f}")
print("Three routes, one line.")

### The likelihood-ratio identity

In [ ]:
(m1, b1), (m2, b2) = lines["model 1"], lines["model 2"]
ratio_direct = np.exp(log_likelihood(m2, b2) - log_likelihood(m1, b1))
ratio_sse = np.exp(-0.5 * (sse(m2, b2) - sse(m1, b1)))
print(f"likelihood ratio model2/model1: {ratio_direct:.3e}")
print(f"e^(-(SSE2 - SSE1)/2):           {ratio_sse:.3e}")
print("Likelihood comparisons ARE squared-error comparisons, re-scaled.")

## 5. Visualizations

### The generative story: Gaussians standing on the road

In [ ]:
def draw_line_with_gaussians(ax, m, b, color, title):
    grid = np.linspace(-0.5, 5.5, 200)
    ax.plot(grid, m * grid + b, color=color, linewidth=2)
    prod_note = 0.0
    for xi, yi in zip(x, y):
        center = m * xi + b
        yy = np.linspace(center - 3, center + 3, 120)
        dens = np.exp(-0.5 * (yy - center)**2) / np.sqrt(2 * np.pi)
        ax.plot(xi + dens * 1.2, yy, color=color, alpha=0.55, linewidth=1)
        ax.plot([xi, xi], [center, yi], color="gray", linewidth=1, linestyle=":")
        prod_note += -0.5 * (yi - center)**2
    ax.scatter(x, y, s=55, color="black", zorder=3)
    ax.set_title(f"{title}\nSSE = {sse(m, b):.1f}, log-lik = {log_likelihood(m, b):.1f}",
                 fontsize=10)
    ax.set_xlim(-0.5, 6.8)
    ax.grid(True, alpha=0.3)

fig, axes = plt.subplots(1, 3, figsize=(14.5, 4.4), sharey=True)
for ax, (name, (m, b)), color in zip(axes, lines.items(),
                                     ["tab:red", "tab:green", "tab:orange"]):
    draw_line_with_gaussians(ax, m, b, color, name)
axes[0].set_ylabel("y")
plt.suptitle("Each line generates points from vertical Gaussians -- the middle road keeps every house in the fat part")
plt.tight_layout()
plt.show()

### One surface, two readings

The log-likelihood surface over $(m, b)$ is the SSE surface flipped upside down — the peak of one is the pit of the other, at the least-squares line.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12.5, 4.8))

cs0 = axes[0].contourf(M, B, ll_grid, levels=30, cmap="viridis")
axes[0].plot(fit[0], fit[1], "r*", markersize=16)
axes[0].set_title("log-likelihood: maximize")
fig.colorbar(cs0, ax=axes[0])

cs1 = axes[1].contourf(M, B, sse_grid, levels=30, cmap="viridis_r")
axes[1].plot(fit[0], fit[1], "r*", markersize=16)
axes[1].set_title("sum of squared errors: minimize")
fig.colorbar(cs1, ax=axes[1])

for ax in axes:
    ax.set_xlabel("slope m")
    ax.set_ylabel("intercept b")
plt.suptitle("Same star: max likelihood = min least squares")
plt.tight_layout()
plt.show()

### Houses along the road: sampling from the winning model

In [ ]:
rng = np.random.default_rng(7)

grid = np.linspace(0, 10, 200)
xs = rng.uniform(0, 10, 60)
ys = m_true * xs + b_true + rng.normal(0, 1, size=60)

plt.figure(figsize=(9, 4.5))
plt.plot(grid, m_true * grid + b_true, color="gold", linewidth=6, alpha=0.8,
         label="the road (the line)")
plt.scatter(xs, ys, s=25, color="tab:brown", label="houses (sampled points)")
plt.title("The generative picture: houses cluster near the road")
plt.xlabel("x")
plt.ylabel("y")
plt.grid(True, alpha=0.3)
plt.legend()
plt.show()

## 6. Exercises

### Basic

**Exercise 1.** Points $(0, 2)$, $(1, 2)$, $(2, 4)$: compute the SSE and the log-likelihood (unit noise) of the lines $y = x + 1$ and $y = x + 2$. Which wins, under both criteria?

**Exercise 2.** In the derivation, three things were discarded: the $\frac{1}{\sqrt{2\pi}}$ factors, the exponential (via log), and the $\frac12$. For each, say in one line why discarding it cannot change the winning line.

### Intermediate

**Exercise 3.** Derive the closed-form least-squares solution by setting the partial derivatives of $\sum(y_i - mx_i - b)^2$ to zero (use sympy on symbolic sums or on the concrete 5-point dataset), and confirm it equals `np.polyfit`.

**Exercise 4.** Repeat the grid search with noise $\sigma = 3$ in the likelihood. Confirm the winning line does not move, and explain which *other* aspect of the surface changes (look at how sharply the likelihood falls off around the peak).

### Advanced

**Exercise 5.** Break the equivalence: use **Laplace** noise (density $\propto e^{-|d|}$) instead of Gaussian and grid-search the maximum-likelihood line for a dataset containing one large outlier. Compare with the least-squares line. Which noise model chases the outlier, and what loss does Laplace-MLE correspond to?

**Exercise 6.** Add the L2 penalty from the last notebook to the negative log-likelihood ($\text{SSE}/2 + \lambda m^2$) and grid-search the winner for $\lambda = 0, 5, 50$. What happens to the slope, and what will the next video call this procedure?

## 7. Solutions

In [ ]:
# Exercise 1: two small lines
pts_x = np.array([0.0, 1.0, 2.0])
pts_y = np.array([2.0, 2.0, 4.0])

for m_, b_ in [(1, 1), (1, 2)]:
    d = pts_y - (m_ * pts_x + b_)
    sse_ = np.sum(d**2)
    ll_ = np.sum(-0.5 * d**2 - 0.5 * np.log(2 * np.pi))
    print(f"y = {m_}x + {b_}: SSE = {sse_:.1f}, log-lik = {ll_:.3f}")
print("y = x + 1 wins on both -- as it must: the two rankings are one ranking.")

In [ ]:
# Exercise 2: why each discard is safe
print("Exercise 2:")
print("1/sqrt(2pi): the same positive constant multiplies every line's")
print("  likelihood -- rescaling all contestants equally moves no ranking.")
print("log: an increasing function preserves order, so argmax is unchanged.")
print("1/2: multiplying the objective by a positive constant stretches the")
print("  axis but leaves the location of the optimum where it was.")

In [ ]:
# Exercise 3: closed form via sympy
import sympy as sp

m_s, b_s = sp.symbols("m b")
loss = sum((yi - (m_s * xi + b_s))**2 for xi, yi in zip(x, y))
sol = sp.solve([sp.diff(loss, m_s), sp.diff(loss, b_s)], [m_s, b_s])
fit = np.polyfit(x, y, 1)
print(f"Exercise 3: sympy: m = {float(sol[m_s]):.4f}, b = {float(sol[b_s]):.4f}")
print(f"np.polyfit:        m = {fit[0]:.4f}, b = {fit[1]:.4f}")

In [ ]:
# Exercise 4: noise scale moves confidence, not the winner
sigma = 3.0
ll_grid3 = -sse_grid / (2 * sigma**2) - len(x) * np.log(sigma * np.sqrt(2 * np.pi))
i3 = np.unravel_index(np.argmax(ll_grid3), ll_grid3.shape)
print(f"Exercise 4: winner with sigma = 3: m = {M[i3]:.3f}, b = {B[i3]:.3f}"
      f"  (same as sigma = 1: m = {M[np.unravel_index(np.argmax(ll_grid), ll_grid.shape)]:.3f})")
drop1 = ll_grid.max() - ll_grid[i_ll[0], i_ll[1] + 40]
drop3 = ll_grid3.max() - ll_grid3[i3[0], i3[1] + 40]
print(f"log-lik drop 0.8 units of b away from the peak: sigma=1: {drop1:.1f}, "
      f"sigma=3: {drop3:.1f}")
print("Bigger noise -> flatter surface: same best line, less certainty about it.")

In [ ]:
# Exercise 5: Laplace noise shrugs at outliers
rng = np.random.default_rng(1)

x_o = np.linspace(0, 5, 12)
y_o = 1.0 * x_o + 1.0 + rng.normal(0, 0.4, 12)
y_o[10] += 12.0                                  # one wild outlier

def best_on_grid(objective):
    vals = objective(M[..., None] * x_o[None, None, :] + B[..., None])
    i = np.unravel_index(np.argmin(vals), vals.shape)
    return M[i], B[i]

gauss_m, gauss_b = best_on_grid(lambda pred: np.sum((y_o - pred)**2, axis=-1))
lap_m, lap_b = best_on_grid(lambda pred: np.sum(np.abs(y_o - pred), axis=-1))

print(f"Exercise 5: least squares (Gaussian MLE): m = {gauss_m:.2f}, b = {gauss_b:.2f}")
print(f"least absolute dev. (Laplace MLE):        m = {lap_m:.2f}, b = {lap_b:.2f}")
print("(true line: m = 1, b = 1)")
print("Squared error panics about the outlier and tilts toward it; |d| charges it")
print("linearly and stays near the truth. Laplace noise <-> L1 (least absolute")
print("deviations) loss: every noise model implies its own loss function.")

In [ ]:
# Exercise 6: penalized likelihood
print("Exercise 6:")
for lam in [0, 5, 50]:
    obj = sse_grid / 2 + lam * M**2
    i = np.unravel_index(np.argmin(obj), obj.shape)
    print(f"  lambda = {lam:>2}: m = {M[i]:.3f}, b = {B[i]:.3f}")
print("The slope shrinks toward 0 as lambda grows (the intercept, unpenalized,")
print("adjusts). Negative log-likelihood + L2 penalty is ridge regression --")
print("and the next video will unmask it as MAP estimation with a Gaussian prior.")

## 8. Conclusion

- The generative model of regression: a line generates points like a **road generates houses** — vertical Gaussians centered on the line, one per $x_i$, each sampled once.
- The dataset's likelihood $\prod \frac{1}{\sqrt{2\pi}} e^{-d_i^2/2}$ collapses under the standard moves (drop constants, factor, take logs) to $-\frac12\sum d_i^2$: **maximizing the likelihood = minimizing the sum of squared distances**. Maximum likelihood *is* least-squares linear regression.
- Consequences verified: likelihood ratios are re-scaled SSE differences; the noise level $\sigma$ changes confidence but never the winning line; and swapping the noise model swaps the loss (Laplace ↔ least absolute deviations — robust to outliers).
- Adding the L2 penalty of the previous notebook to the negative log-likelihood shrinks the slope — ridge regression, about to be revealed as **MAP estimation** in the next video.